In [31]:
# Load required libraries:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime as dt
import plotly.express as px
import random
import matplotlib.pyplot as plt
import talib as ta
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import Dataset



Downloading assets data

In [32]:
sector_mix=['AAPL','MSFT','GOOGL','AMZN','TSLA','JNJ','PG','XOM','CVX','JPM','BAC','WMT','KO','NVDA','META','PFE','UNH','V','HD','NFLX','SBUX','NEE','MRK','GAP','AVGO','BA','T','GM','COST','ADI','DOW','GS','CAT','CRM','AMAT','NKE','GE','TWLO','PYPL']

start = dt.date.today() - dt.timedelta(days=365 * 5)
end = dt.date.today()
Tickers=sector_mix
stock_df = yf.download(Tickers,start,end, progress=False)
returns=stock_df['Close'].pct_change().dropna()

gspc_vix=['^GSPC','^VIX']
market_df=yf.download(gspc_vix, start=start, end=end, progress=False)
market_returns=market_df['Close'].pct_change().dropna()



In [33]:
px.line(market_returns, title='Stock Returns Over Time', labels={'value': 'Returns', 'Date': 'Date'}, template='plotly_dark')


fonctions to compute cvar and generate action labels

In [34]:

# Function to compute  Conditional Value at Risk (CVaR):
def calculate_cvar(data, window=1, alpha=0.05):
    """
    Compute CVaR over a rolling window.
    :param data: DataFrame with stock returns
    :param window: Rolling window size (e.g., 10 days)
    :param alpha: Confidence level (e.g., 0.05 for 95% CVaR)
    :return: DataFrame with CVaR values
    """
    cvar = data.rolling(window=window).apply(
        lambda x: x[x <= np.percentile(x, alpha * 100)].mean(), raw=True
    )
    return cvar



# Function to generate (Buy,Hold, Sell) labels:
    
def generate_action_labels(cvar_10_day, cvar_30_day):
    # Align DataFrames to ensure proper structure:
    cvar_10_day, cvar_30_day = cvar_10_day.align(cvar_30_day, join='inner', axis=0)

    # Initialize action labels and counts:
    action_labels = pd.DataFrame(index=cvar_10_day.index, columns=cvar_10_day.columns)
    action_counts = {ticker: {0: 0, 1: 0, 2: 0} for ticker in cvar_10_day.columns}

    for date in cvar_10_day.index:
        for ticker in cvar_10_day.columns:
            try:
                # Check for NaNs and assign actions:
                if pd.isna(cvar_10_day.loc[date, ticker]) or pd.isna(cvar_30_day.loc[date, ticker]):
                    action = np.nan
                elif cvar_10_day.loc[date, ticker] < cvar_30_day.loc[date, ticker]:
                    action = 1  # Buy
                elif cvar_10_day.loc[date, ticker] > cvar_30_day.loc[date, ticker]:
                    action = 2 # Sell
                else:
                    action = 0  # Hold

                # Set the action label:
                action_labels.loc[date, ticker] = action

                # Update action counts:
                if not pd.isna(action):
                    action_counts[ticker][int(action)] += 1
            except KeyError:
                action_labels.loc[date, ticker] = np.nan  

    return action_labels.astype(float), action_counts
# Function to calculate technical indicators to be used as features:


fonction to compute indicator from tickers

In [35]:

def calculate_indicators(tickers, start, end):
    data = {}  # Dictionary to store indicators for each ticker

    for ticker in tickers:
        try:
            stock_data = yf.download(ticker, start=start, end=end, progress=False)

            close_prices = stock_data['Close'].astype(np.float64).values.flatten()
            high_prices = stock_data['High'].astype(np.float64).values.flatten()
            low_prices = stock_data['Low'].astype(np.float64).values.flatten()
            volume = stock_data['Volume'].astype(np.float64).values.flatten()

            # Add indicators for the current ticker:
            data[ticker] = {
                "MACD": ta.MACD(close_prices)[0].tolist(),
                "RSI": ta.RSI(close_prices).tolist(),
                "CCI": ta.CCI(high_prices, low_prices, close_prices).tolist(),
                "ADX": ta.ADX(high_prices, low_prices, close_prices).tolist(),
                "STOCH_K": ta.STOCH(high_prices, low_prices, close_prices)[0].tolist(),
                "WILLR": ta.WILLR(high_prices, low_prices, close_prices).tolist(),
                "BB_Upper": ta.BBANDS(close_prices)[0].tolist(),
                "BB_Middle": ta.BBANDS(close_prices)[1].tolist(),
                "BB_Lower": ta.BBANDS(close_prices)[2].tolist(),
                "MFI": ta.MFI(high_prices, low_prices, close_prices, volume).tolist(),
                "EMA": ta.EMA(close_prices).tolist(),
                "ATR": ta.ATR(high_prices, low_prices, close_prices).tolist(),
                "SAR": ta.SAR(high_prices, low_prices).tolist(),
                "OBV": ta.OBV(close_prices, volume).tolist(),
            }
        except Exception as e:
            print(f"Error downloading data for {ticker}: {e}")

    # Convert the data dictionary to a DataFrame:
    indicators_df = pd.DataFrame(data).transpose()  # Transpose for readability
    return indicators_df

indicators_df = calculate_indicators(Tickers, start, end)



data clearning function

In [36]:

def align_and_fill_indicators(market_returns, indicators_df):

	# Fill NaN values in lists within columns:
	def fill_na_in_column(col):
		return col.map(lambda x: np.nan_to_num(x, nan=np.nanmean(x)) if isinstance(x, np.ndarray) else x)

	indicators_df_cleaned = indicators_df.apply(fill_na_in_column, axis=0)
	uniform_length = min(len(vals) for ticker in indicators_df_cleaned.index for vals in indicators_df_cleaned.loc[ticker].values)


	flattened_indicators = {}
	for ticker in indicators_df_cleaned.index:
		flattened_indicators[ticker] = pd.DataFrame(
			{col: np.array(vals[:uniform_length]) if len(vals) > uniform_length else np.pad(vals, (0, uniform_length - len(vals)))
			for col, vals in indicators_df_cleaned.loc[ticker].items()}
		)



	# Rebuild combined_states :
	combined_states = {}
	for ticker in indicators_df_cleaned.index:
		ticker_data = flattened_indicators[ticker]
		#print("Ticker data shape:", ticker_data.shape)

		# Ensure proper alignment and tiling of market_returns features:
		aligned_market_returns = pd.DataFrame(
			np.tile(market_returns.iloc[:ticker_data.shape[0]].values, (1, 1)),
			columns=market_returns.columns
		)
		print("Aligned market returns shape:", aligned_market_returns.shape)
		# Combine ticker-specific indicators with market-wide returns:
		combined_states[ticker] = pd.concat([ticker_data, aligned_market_returns], axis=1)

	for ticker in combined_states.keys():
		combined_states[ticker] = combined_states[ticker].fillna(combined_states[ticker].mean())

	# Drop rows with NaNs in each ticker's DataFrame:
	for ticker in combined_states.keys():
		combined_states[ticker] = combined_states[ticker].dropna()

	for ticker, df in combined_states.items():
		print(f"{ticker} - Remaining NaN count:", df.isna().sum().sum())

		print(f"{ticker} - Shape after dropna:", df.shape)

	return combined_states


combined_states = align_and_fill_indicators(market_returns, indicators_df)


Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)
Aligned market returns shape: (1250, 2)


In [37]:
combined_states['NVDA']

,MACD,RSI,CCI,ADX,STOCH_K,WILLR,BB_Upper,BB_Middle,BB_Lower,MFI,EMA,ATR,SAR,OBV,^GSPC,^VIX
0,0.998633,54.067062,16.055329,24.998197,54.755590,-43.312021,108.153054,99.464529,90.776004,52.340413,99.250324,3.728634,98.611643,5.338670e+08,-0.003197,0.070352
1,0.998633,54.067062,16.055329,24.998197,54.755590,-43.312021,108.153054,99.464529,90.776004,52.340413,99.250324,3.728634,34.495068,-2.204680e+08,0.001657,0.010955
2,0.998633,54.067062,16.055329,24.998197,54.755590,-43.312021,108.153054,99.464529,90.776004,52.340413,99.250324,3.728634,34.495068,-7.526310e+08,0.002294,-0.041280
3,0.998633,54.067062,16.055329,24.998197,54.755590,-43.312021,108.153054,99.464529,90.776004,52.340413,99.250324,3.728634,34.495068,-3.174690e+08,-0.022725,0.540366
4,0.998633,54.067062,16.055329,24.998197,54.755590,-43.312021,108.153054,99.464529,90.776004,52.340413,99.250324,3.728634,34.345049,-6.005380e+08,0.013200,-0.197764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1248,-12.119221,28.739661,-116.873243,36.838781,18.276446,-85.508326,236.127109,203.804501,171.481894,29.766308,205.452291,8.695363,189.843982,2.414627e+10,-0.019186,0.187047
1249,-13.235041,25.992974,-138.668323,39.491599,23.626918,-75.572414,236.828076,201.252502,165.676927,24.711079,202.919240,9.854979,185.053185,2.339856e+10,-0.010255,0.111305
1250,-14.224871,24.859579,-106.479215,41.505162,36.231207,-80.264078,237.432655,198.688502,159.944348,29.397151,200.334129,9.980338,177.802549,2.303276e+10,0.000416,0.003347
1251,-15.373543,22.716545,-110.741043,43.476552,36.136498,-89.689569,237.573238,195.641002,153.708766,25.213816,197.482894,9.852457,172.002040,2.269894e+10,0.000416,0.003347


computing cvar on cloasae price data

In [38]:


sequence_length = 10
def calculate_cvar(data, window=None, alpha=0.05):
    """
    Compute CVaR either over a rolling window or the entire dataset if window is None.
    :param data: DataFrame with stock returns
    :param window: Rolling window size (e.g., 10 days), or None for full dataset CVaR
    :param alpha: Confidence level (e.g., 0.05 for 95% CVaR)
    :return: DataFrame or scalar with CVaR values
    """
    if window is None:
        # Calculate CVaR for the entire dataset
        cvar = data.apply(lambda x: x[x <= np.percentile(x, alpha * 100)].mean() if len(x[x <= np.percentile(x, alpha * 100)])>0 else 0, raw=True)
    else:
        # Calculate rolling CVaR
        cvar = data.rolling(window=window).apply(
            lambda x:x[x <= np.percentile(x, alpha * 100)].mean() if len(x[x <= np.percentile(x, alpha * 100)])>0 else 0, raw=True)
    return cvar
    
cvar_df=calculate_cvar(returns, window=1, alpha=0.05).bfill()

cvar_10_day=calculate_cvar(returns, window=10, alpha=0.05).bfill()
cvar_30_day=calculate_cvar(returns, window=30, alpha=0.05).bfill()


In [39]:

# Display first few rows of the smoothed CVaR

action_df, action_counts=generate_action_labels(cvar_10_day, cvar_30_day)
counts_df = pd.DataFrame(action_counts).T
counts_df.columns = ['Hold (0)', 'Buy (1)', 'Sell (2)']
print(" 10 day and 30 day CVar:",counts_df)

        


 10 day and 30 day CVar:        Hold (0)  Buy (1)  Sell (2)
AAPL          0      454       798
ADI           0      464       788
AMAT          0      358       894
AMZN          0      451       801
AVGO          0      430       822
BA            0      480       772
BAC           0      433       819
CAT           0      410       842
COST          0      410       842
CRM           0      405       847
CVX           0      400       852
DOW           0      416       836
GAP           0      401       851
GE            0      439       813
GM            0      409       843
GOOGL         0      432       820
GS            0      463       789
HD            0      436       816
JNJ           0      435       817
JPM           0      430       822
KO            0      415       837
META          0      447       805
MRK           0      394       858
MSFT          0      431       821
NEE           0      433       819
NFLX          0      378       874
NKE           0      400      

In [40]:
action_counts

{'AAPL': {0: 0, 1: 454, 2: 798},
 'ADI': {0: 0, 1: 464, 2: 788},
 'AMAT': {0: 0, 1: 358, 2: 894},
 'AMZN': {0: 0, 1: 451, 2: 801},
 'AVGO': {0: 0, 1: 430, 2: 822},
 'BA': {0: 0, 1: 480, 2: 772},
 'BAC': {0: 0, 1: 433, 2: 819},
 'CAT': {0: 0, 1: 410, 2: 842},
 'COST': {0: 0, 1: 410, 2: 842},
 'CRM': {0: 0, 1: 405, 2: 847},
 'CVX': {0: 0, 1: 400, 2: 852},
 'DOW': {0: 0, 1: 416, 2: 836},
 'GAP': {0: 0, 1: 401, 2: 851},
 'GE': {0: 0, 1: 439, 2: 813},
 'GM': {0: 0, 1: 409, 2: 843},
 'GOOGL': {0: 0, 1: 432, 2: 820},
 'GS': {0: 0, 1: 463, 2: 789},
 'HD': {0: 0, 1: 436, 2: 816},
 'JNJ': {0: 0, 1: 435, 2: 817},
 'JPM': {0: 0, 1: 430, 2: 822},
 'KO': {0: 0, 1: 415, 2: 837},
 'META': {0: 0, 1: 447, 2: 805},
 'MRK': {0: 0, 1: 394, 2: 858},
 'MSFT': {0: 0, 1: 431, 2: 821},
 'NEE': {0: 0, 1: 433, 2: 819},
 'NFLX': {0: 0, 1: 378, 2: 874},
 'NKE': {0: 0, 1: 400, 2: 852},
 'NVDA': {0: 0, 1: 390, 2: 862},
 'PFE': {0: 0, 1: 397, 2: 855},
 'PG': {0: 0, 1: 428, 2: 824},
 'PYPL': {0: 0, 1: 405, 2: 847},
 'S

In [41]:
# Construct dataset using combined_states and action_labels:
action_labels, action_counts = generate_action_labels(cvar_10_day, cvar_30_day)
# 1) Construire feature_data AVEC les dates
dates = stock_df.index  # mêmes jours de bourse que les indicateurs
for t, df in combined_states.items():
    assert len(df) == len(dates), (t, len(df), len(dates))
    df.index = pd.Index(dates, name="Date")

feature_data = (
    pd.concat(combined_states.values(), keys=combined_states.keys(), names=["Ticker", "Date"])
    .reset_index()
)

# 2) Retirer les seq_len premières lignes DE CHAQUE ticker
seq_len = 10
trimmed_feature_data = feature_data[feature_data.groupby("Ticker").cumcount() >= seq_len]

# 3) Labels et CVaR en format long (Date, Ticker, valeur)
reshaped_action_labels = action_labels.stack().rename("Action").reset_index()
reshaped_action_labels.columns = ["Date", "Ticker", "Action"]

reshaped_cvar_df = cvar_df.stack().rename("CVaR").reset_index()
reshaped_cvar_df.columns = ["Date", "Ticker", "CVaR"]

# 4) Merge sur (Date, Ticker) : chaque ligne reçoit le label de SON jour
merged_data = (
    trimmed_feature_data
    .merge(reshaped_action_labels, on=["Date", "Ticker"], how="inner", validate="one_to_one")
    .merge(reshaped_cvar_df,       on=["Date", "Ticker"], how="inner", validate="one_to_one")
)

print("Shape of merged_data:", merged_data.shape)
print(merged_data.head())


Shape of merged_data: (48477, 20)
  Ticker       Date     MACD        RSI         CCI        ADX    STOCH_K  \
0   AAPL 2021-12-06  1.01368  53.547828   15.770037  24.948365  48.521574   
1   AAPL 2021-12-07  1.01368  53.547828   15.770037  24.948365  63.192470   
2   AAPL 2021-12-08  1.01368  53.547828   15.770037  24.948365  84.137263   
3   AAPL 2021-12-09  1.01368  53.547828  167.373594  24.948365  93.130562   
4   AAPL 2021-12-10  1.01368  75.457929  160.984247  24.948365  93.707269   

       WILLR    BB_Upper   BB_Middle    BB_Lower        MFI         EMA  \
0 -43.106859  218.764084  207.197813  195.631543  54.797087  206.782861   
1 -43.106859  218.764084  207.197813  195.631543  54.797087  206.782861   
2 -43.106859  218.764084  207.197813  195.631543  54.797087  206.782861   
3 -10.740589  218.764084  207.197813  195.631543  54.797087  206.782861   
4  -0.773563  218.764084  207.197813  195.631543  81.266166  206.782861   

        ATR         SAR          OBV     ^GSPC      

In [42]:


# Identify feature columns: 
feature_columns = [col for col in merged_data.columns if col not in ['Date', 'Ticker', 'Action']]

# Initialize the scaler:
scaler = StandardScaler()

# Fit and transform the feature columns:
merged_data[feature_columns] = scaler.fit_transform(merged_data[feature_columns])


In [43]:
feature_df       = merged_data[["Date", "Ticker"] + feature_columns]
action_labels_df = merged_data[["Date", "Ticker", "Action"]]
aligned_cvar_df  = merged_data[["Date", "Ticker", "CVaR"]]


assert len(feature_df) == len(action_labels_df) == len(aligned_cvar_df)
aligned_cvar_df.head()


,Date,Ticker,CVaR
0,2021-12-06,AAPL,0.939592
1,2021-12-07,AAPL,1.566894
2,2021-12-08,AAPL,0.997163
3,2021-12-09,AAPL,-0.161471
4,2021-12-10,AAPL,1.232486


In [44]:
# Build DRL models:
class DQNDataset(Dataset):
    def __init__(self, states, actions, rewards, next_states, dones):
        self.states = states
        self.actions = actions
        self.rewards = rewards
        self.next_states = next_states
        self.dones = dones

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return (
            self.states[idx],
            self.actions[idx],
            self.rewards[idx],
            self.next_states[idx],
            self.dones[idx]
        )
    


class CVaRDataset(Dataset):
    def __init__(self, feature_df, action_labels_df, aligned_cvar_df, sequence_length=10):
        self.sequence_length = sequence_length
        indicator_columns = [col for col in feature_df.columns if col not in ("Date", "Ticker")]

        # Format large : une ligne par date, les tickers côte à côte
        features_by_date = (
            feature_df
            .pivot(index="Date", columns="Ticker", values=indicator_columns)
            .sort_index()
        )
        trading_dates = features_by_date.index

        actions_by_date = (
            action_labels_df
            .pivot(index="Date", columns="Ticker", values="Action")
            .reindex(trading_dates)
        )
        cvar_by_date = (
            aligned_cvar_df
            .pivot(index="Date", columns="Ticker", values="CVaR")
            .reindex(trading_dates)
        )

        self.tickers = list(actions_by_date.columns)
        self.n_tickers = len(self.tickers)

        # Conversion en tableaux numpy float32
        feature_matrix = features_by_date.to_numpy(dtype=np.float32)          # (n_dates, n_indicators * n_tickers)
        action_matrix = actions_by_date.fillna(0).to_numpy(dtype=np.float32)  # (n_dates, n_tickers)
        cvar_matrix = cvar_by_date.fillna(0).to_numpy(dtype=np.float32)       # (n_dates, n_tickers)

        # Normalisation min-max de la CVaR sur tout le dataset
        epsilon = 1e-6
        cvar_min = cvar_matrix.min()
        cvar_max = cvar_matrix.max()
        normalized_cvar_matrix = (cvar_matrix - cvar_min) / (cvar_max - cvar_min + epsilon)

        self.states = []
        self.actions = []
        self.rewards = []
        self.next_states = []

        # Fenêtre glissante : l'état = les `sequence_length` jours précédant `current_day`
        for current_day in range(sequence_length, len(feature_matrix) - 1):
            window_start = current_day - sequence_length

            state_sequence = feature_matrix[window_start:current_day]
            next_state_sequence = feature_matrix[window_start + 1:current_day + 1]
            daily_actions = action_matrix[current_day]
            daily_normalized_cvar = normalized_cvar_matrix[current_day]

            reward = self.calculate_reward(daily_actions, daily_normalized_cvar)
            if np.any(np.isnan(reward)):
                continue

            self.states.append(state_sequence)
            self.next_states.append(next_state_sequence)
            self.actions.append(daily_actions)
            self.rewards.append(reward)


    # cvar_values here are CVaR with window=1 i.e. cvar_values==returns:

    def calculate_reward(self, action, cvar_values):
        """
        Calculate reward based on the action and the corresponding normalized CVaR values.
        """
        # Normalize the CVaR values
    
        # Define weights based on CVaR values
        buy_weight = np.where(cvar_values<0,1.5,1)
        sell_weight = np.where(cvar_values>0,1.2,1)
        hold_weight = 0.8

        # Calculate rewards
        reward = -buy_weight * abs(cvar_values) * (action == 1)  # Penalize buy action-move gainst trend
        reward += sell_weight * abs(cvar_values) * (action == 2)  # Reward sell action on gains
        reward += -hold_weight * abs(cvar_values) * (action == 0)  # Penalize holding slightly based on risk

        # Optional: Clip the reward to avoid extreme values
        reward = np.clip(reward, -3, 3)

        return reward
    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        state = np.array(self.states[idx], dtype=np.float32)
        action = np.array(self.actions[idx], dtype=np.float32)
        reward = np.array(self.rewards[idx], dtype=np.float32)
        next_state = np.array(self.next_states[idx], dtype=np.float32)

        done_flags = np.zeros(len(action), dtype=np.float32)
        if idx == len(self.states) - 1:
            done_flags[:] = 1.0

        return (
            torch.tensor(state, dtype=torch.float32),
            torch.tensor(action, dtype=torch.float32),
            torch.tensor(reward, dtype=torch.float32),
            torch.tensor(next_state, dtype=torch.float32),
            torch.tensor(done_flags, dtype=torch.float32)
        )
        


In [45]:
feature_df

,Date,Ticker,MACD,RSI,CCI,ADX,STOCH_K,WILLR,BB_Upper,BB_Middle,BB_Lower,MFI,EMA,ATR,SAR,OBV,^GSPC,^VIX,CVaR
0,2021-12-06,AAPL,0.065137,0.125411,0.053911,0.077838,-0.155351,0.109106,0.104472,0.114107,0.124581,0.163965,0.113594,-0.050259,-0.185479,-0.082424,1.874823,-2.440198,0.939592
1,2021-12-07,AAPL,0.065137,0.125411,0.053911,0.077838,0.414292,0.109106,0.104472,0.114107,0.124581,0.163965,0.113594,-0.050259,-0.182728,-0.042001,0.244967,-1.158865,1.566894
2,2021-12-08,AAPL,0.065137,0.125411,0.053911,0.077838,1.227538,0.109106,0.104472,0.114107,0.124581,0.163965,0.113594,-0.050259,-0.178343,-0.002721,-0.704579,1.007148,0.997163
3,2021-12-09,AAPL,0.065137,0.125411,1.447419,0.077838,1.576731,1.158320,0.104472,0.114107,0.124581,0.163965,0.113594,-0.050259,-0.170920,-0.039290,0.842809,-1.690219,-0.161471
4,2021-12-10,AAPL,0.065137,1.880856,1.388689,0.077838,1.599123,1.481420,0.104472,0.114107,0.124581,1.804994,0.113594,-0.035211,-0.161949,-0.000546,-0.885406,1.035009,1.232486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48472,2026-11-11,PYPL,0.078373,0.640202,0.350034,-0.521186,0.177989,0.816459,-0.740191,-0.731907,-0.720202,0.888053,-0.735977,-0.667968,-0.732725,-0.352822,-1.814948,2.274956,0.507940
48473,2026-11-12,PYPL,0.060771,0.153486,0.172886,-0.635508,-0.486415,0.230671,-0.739899,-0.731585,-0.719847,0.945546,-0.735586,-0.673291,-0.730956,-0.354887,-0.988925,1.339256,-0.920377
48474,2026-11-13,PYPL,0.058811,0.460319,0.335051,-0.731224,-0.296418,0.717172,-0.739069,-0.730855,-0.719231,1.398809,-0.734861,-0.678651,-0.729328,-0.351694,-0.001952,0.005562,0.728426
48475,2026-11-16,PYPL,0.048957,0.277822,0.215084,-0.804676,-0.447803,0.306308,-0.738723,-0.730198,-0.718227,0.884426,-0.734347,-0.683838,-0.727831,-0.354806,-0.001952,0.005562,-0.369359


In [46]:

# Initialize the Dataset:

sequence_length = 10 
full_dataset = CVaRDataset(feature_df, action_labels_df, aligned_cvar_df, sequence_length=10)

# Split the dataset into train and test indices:

train_indices, test_indices = train_test_split(
    list(range(len(full_dataset))),
    test_size=0.1,
    random_state=42
)

# Split into train and test subsets:
train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
test_dataset = torch.utils.data.Subset(full_dataset, test_indices)
print("Number of training samples:", len(train_indices))
print("Number of testing samples:", len(test_indices))

# Create data loaders:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

input_shape = full_dataset[0][0].shape  # Get the shape of the input state
n_tickers = len(Tickers)


Number of training samples: 1108
Number of testing samples: 124
